In [ ]:
%%javascript(() => {  // 只隐藏编辑器，不隐藏 cell 的 toolbar / prompt  const selectors = ['.jp-InputArea-editor', '.cm-editor', '.CodeMirror'];  // 找到当前 Notebook 使用的编辑器 DOM（优先匹配第一个存在的 selector）  function getEditors() {    for (const s of selectors) {      const nodes = document.querySelectorAll(s);      if (nodes.length) return { sel: s, nodes };    }    return { sel: null, nodes: [] };  }  // 切换显示/隐藏  function toggle() {    const { sel, nodes } = getEditors();    if (!nodes.length) return alert('没找到编辑器区域：' + selectors.join(' / '));    const hide = nodes[0].style.display !== 'none';    nodes.forEach(n => n.style.display = hide ? 'none' : '');    // 仅用于调试：输出当前使用的 selector    console.log("toggle selector:", sel);  }  // 创建右上角按钮（避免重复创建）  let btn = document.getElementById('toggleCodeBtn');  if (!btn) {    btn = document.createElement('button');    btn.id = 'toggleCodeBtn';    btn.textContent = 'Hide/Show Code';    btn.style.cssText =      'position:fixed;top:12px;right:12px;z-index:99999;padding:6px 12px;border-radius:6px;';    btn.addEventListener('click', toggle);    document.body.appendChild(btn);  }  // 默认隐藏编辑器（只隐藏代码，不影响按钮/工具栏/输出）  const { nodes } = getEditors();  nodes.forEach(n => n.style.display = 'none');})();

# IBD反转信号识别分析

**目标**: 基于IBD方法论识别市场反转点

**核心概念**:
- 跟踪日 (Follow-Through Day): 市场底部反转信号
- 分布日 (Distribution Day): 市场顶部预警信号
- 市场状态: Confirmed Uptrend / Uptrend Under Pressure / Market in Correction / Rally Attempt

**参考**: A股市场趋势环境判别与预测方法研究.pdf

In [ ]:
# 统一环境初始化（自动检测项目路径）from notebooks.lib import (    setup_research_environment,    ErrorBoundary,    ResultSaver)# 初始化研究环境env = setup_research_environment(verbose=True)# 导入必要的库import pandas as pdimport numpy as npfrom datetime import datetime, timedelta# 从环境获取组件jq = Nonetry:    jq = env.get_jqdata_client()except Exception as e:    print(f"⚠️ JQData初始化失败: {{e}}")# 加载配置config = env.load_config('config')INDEX_CODE = config.get('data', {{}}).get('default_index', '000001.XSHG')# 初始化结果保存器result_saver = ResultSaver("ibd_reversal_analysis")print('✅ 环境加载完成')

## 1. IBD方法论介绍

### 跟踪日 (Follow-Through Day)
- 在市场低点后4-7个交易日
- 当日涨幅 > 1.7%（A股本土化调整）
- 成交量高于前一日
- 标志着市场底部反转确认

### 分布日 (Distribution Day)
- 当日跌幅 > 0.2%（A股本土化）
- 成交量高于前一日
- 25个交易日内累计5个以上分布日为危险信号
- 标志着机构出货

In [ ]:
# 执行IBD分析
index_code = "000001.XSHG"
result = ibd.analyze(index_code=index_code, lookback_days=60)

print("=" * 60)
print("IBD分析结果")
print("=" * 60)
print(f"分析日期: {result.analysis_date}")
print(f"市场状态: {result.market_status.value}")
print(f"分布日数量: {result.distribution_count}")
print(f"跟踪日数量: {len(result.follow_through_days)}")
print(f"\n交易建议: {result.recommendation}")

In [ ]:
# 详细分布日列表
if result.distribution_days:
    print("\n📉 分布日详情:")
    for i, dd in enumerate(result.distribution_days[-10:], 1):
        status = "✓" if not dd.expired else "(已过期)"
        print(f"  {i}. {dd.date}: 跌幅{dd.loss_pct:.2f}%, 量比{dd.volume_ratio:.2f} {status}")
else:
    print("\n无分布日")

In [ ]:
# 跟踪日详情
if result.follow_through_days:
    print("\n📈 跟踪日详情:")
    for i, ftd in enumerate(result.follow_through_days, 1):
        valid = "✓ 有效" if ftd.is_valid else "✗ 无效"
        print(f"  {i}. {ftd.date}: 涨幅{ftd.gain_pct:.2f}%, 量比{ftd.volume_ratio:.2f}, 距低点{ftd.days_from_low}天 {valid}")
else:
    print("\n无跟踪日")

## 2. 市场状态可视化

In [ ]:
# 获取指数数据
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=120)).strftime('%Y-%m-%d')

df = jq.get_price(index_code, start_date=start_date, end_date=end_date, 
                  frequency='daily', fields=['close', 'volume'])
df = df.reset_index()
df.columns = ['date', 'close', 'volume']

# 计算移动平均
df['ma20'] = df['close'].rolling(20).mean()
df['ma50'] = df['close'].rolling(50).mean()

print(f"数据范围: {df['date'].iloc[0]} 至 {df['date'].iloc[-1]}")
print(f"数据条数: {len(df)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# 价格图
ax1 = axes[0]
ax1.plot(df['date'], df['close'], label='收盘价', color='blue', linewidth=1.5)
ax1.plot(df['date'], df['ma20'], label='MA20', color='orange', linestyle='--')
ax1.plot(df['date'], df['ma50'], label='MA50', color='red', linestyle='--')

# 标注分布日
for dd in result.distribution_days:
    try:
        dd_date = pd.to_datetime(dd.date)
        if dd_date in df['date'].values:
            idx = df[df['date'] == dd_date].index[0]
            ax1.scatter(dd_date, df.loc[idx, 'close'], color='red', s=100, marker='v', zorder=5)
    except:
        pass

# 标注跟踪日
for ftd in result.follow_through_days:
    try:
        ftd_date = pd.to_datetime(ftd.date)
        if ftd_date in df['date'].values:
            idx = df[df['date'] == ftd_date].index[0]
            ax1.scatter(ftd_date, df.loc[idx, 'close'], color='green', s=100, marker='^', zorder=5)
    except:
        pass

ax1.set_title(f'{index_code} 价格与IBD信号')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 成交量图
ax2 = axes[1]
colors = ['green' if df.loc[i, 'close'] >= df.loc[i-1, 'close'] else 'red' 
          for i in range(1, len(df))]
colors.insert(0, 'gray')
ax2.bar(df['date'], df['volume'], color=colors, alpha=0.6)
ax2.set_title('成交量')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. A股本土化改进验证

根据研究文献，A股市场的IBD信号需要本土化调整：
- 跟踪日涨幅阈值：1.7%（美股2%）
- 分布日跌幅阈值：0.2%（美股0.2%，保持一致）
- 窗口期：25个交易日（与美股一致）

In [ ]:
# 多指数对比分析
indices = {
    '000001.XSHG': '上证指数',
    '399001.XSHE': '深证成指', 
    '399006.XSHE': '创业板指'
}

print("=" * 70)
print("多指数IBD分析对比")
print("=" * 70)
print(f"{'指数':<15} {'状态':<20} {'分布日':<8} {'跟踪日':<8}")
print("-" * 70)

for code, name in indices.items():
    try:
        r = ibd.analyze(index_code=code, lookback_days=60)
        print(f"{name:<15} {r.market_status.value:<20} {r.distribution_count:<8} {len(r.follow_through_days):<8}")
    except Exception as e:
        print(f"{name:<15} 分析失败: {str(e)[:30]}")

## 4. 保存研究结论

In [ ]:
conclusion = {
    "analysis_date": result.analysis_date,
    "market_status": result.market_status.value,
    "distribution_count": result.distribution_count,
    "follow_through_count": len(result.follow_through_days),
    "recommendation": result.recommendation
}

save_research_conclusion(
    module="ibd_reversal_analysis",
    findings=conclusion,
    recommendation=result.recommendation,
    metadata={"index_code": index_code}
)

print("✅ 研究结论已保存")